## Import packages

See YAML file for specific package requirements

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib qt
import cv2
from tensorflow.keras.preprocessing.image import load_img
from keras.saving import load_model
import segmenteverygrain as seg
import sez
from importlib import reload
from segment_anything import sam_model_registry, SamPredictor


## Setting up and loading models/checkpoints

Insert the path to your model, model checkpoints, and the image you would like to segment. The SAM model checkpoints can be downloaded from : https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

In [ ]:
#path to where your segmenteveryzircon model is stored
model_fname = "sez_model.keras"
# SAM model checkpoints
checkpoint = "/Users/omw339/Downloads/sam_vit_h_4b8939.pth"
#path of the image you would like to segment
image_fname = 'example_images/example_input_image.png' #replace with your image path

load model and SAM checkpoints

In [ ]:
model = load_model(model_fname, custom_objects={'weighted_crossentropy': seg.weighted_crossentropy})
sam = sam_model_registry["default"](checkpoint=checkpoint)

## Run segmentation

The following cell runs the segmentation model on the specified image to produce an inital output of grain geometries. 

For high resolution, large, images, this process may take a while. It may be necessary to [downsample](https://visionbook.mit.edu/upsamplig_downsampling_2.html) your image depending on the size of the image and the computing resources available to you.

In [ ]:
all_grains, image_pred, all_coords = sez.predict_large_image(image_fname, model, sam, min_area=400.0, patch_size=2000, overlap=200, remove_large_objects=True)

## Plot initial prediction image and Initialize grain list/labels

In [ ]:
image = np.array(load_img(image_fname))
fig, ax = plt.subplots(figsize=(15,10))
ax.set_autoscale_on(False) 
plt.xticks([])
plt.yticks([])
seg.plot_image_w_colorful_grains(image, all_grains, ax, cmap='Paired')
plt.axis('equal')
plt.xlim([0, np.shape(image)[1]])
plt.ylim([np.shape(image)[0], 0]);

In [ ]:
all_grains, labels, pred_mask = seg.get_grains_from_patches(ax, image)
initial_n = len(all_grains)

## Adding and Deleting Grains
- Run the below cell to activate the interative SEZ segmentation GUI. Here, you will be able to add, delete, and split grains.

Type one of the following letters to activate the method of choice.
- **a** : add grains
- **d** : delete grains
- **i** : click to activate splitting grains
- **g** : toggle off segmented polygons for viewing original image


Then, with your mouse, left click to do the chosen method. If you are splitting grains, left click three times to form a
line that you want to split on, and right click when you are done.

In [ ]:
reload(seg)

predictor = SamPredictor(sam)
predictor.set_image(image)

coords = []
grain_inds = []
all_grains = []
ax.set_autoscale_on(False)


cid_mouse = fig.canvas.mpl_connect(
    'button_press_event',
    lambda e: sez.unified_click(e, ax, image, predictor, all_grains, grain_inds)
)

cid_key = fig.canvas.mpl_connect(
    'key_press_event',
    lambda e: sez.unified_key(e, ax, fig, all_grains, grain_inds)
)


Run this cell to close the interactive GUI

In [ ]:
# Disconnect event handlers
fig.canvas.mpl_disconnect(cid_mouse)
fig.canvas.mpl_disconnect(cid_key)
all_grains, labels, gt_mask = seg.get_grains_from_patches(ax, image)

Use this function to update the 'labels' array after deleting and merging grains (the 'all_grains' list is updated when doing the deletion and merging):

plot image after initial deletions

In [ ]:
image = np.array(load_img(image_fname))
fig, ax = plt.subplots(figsize=(15,10))
ax.set_autoscale_on(False)
ax.set_xlim(0, image.shape[1])
ax.set_ylim(image.shape[0], 0)
plt.xticks([])
plt.yticks([])
seg.plot_image_w_colorful_grains(image, all_grains, ax, cmap='Paired')
plt.axis('equal')

In [ ]:
final_n = len(all_grains)
print(final_n)

### Creating Metadata Table for Segmentation Results

In [ ]:
df_meta = sez.create_metadata_table(image_fname, final_n, model_fname, save_csv=False)

After you are done with the deletion / addition of grain masks, run this cell to generate an updated set of grains:

## Last Steps: Save mask, grain polygons, and grain coordinates
- These steps are vital if you would like to come back to your work later 
- The below command creates the binary mask associated with the image you just segmented. 
**Note:** When you open the file it will appear black, that is ok. The line should return 'True' meaning that the file was created. You can double check this by going into the folder where you saved the file.

### Saving binary mask 

In [ ]:
#saving binary mask
cv2.imwrite('/Users/omw339/Desktop/practice_mask.png', gt_mask)

### Saving Grain Coordinates 
### NOTE: Essential for extracting measurements
- Run the following cells to create a geopandas geodataframe to store all of the polygons with their respective geometries and coordinates

In [ ]:
gdf = sez.grains_to_geodataframe(image_fname, all_grains)
gdf.head()

saving the csv file with the grain coordinates

In [ ]:
gdf.to_csv('test_coordinates.csv')

You are now done creating the grain polygons and masks. Use the morphometrics.ipynb to create the morphology dataset

## Steps to Re-Read your segmented grains back after finishing segmentation

### Step 1: Load polygons into geodataframe and re-initialize all_grains

In [ ]:
# Replace 'your_polygons.csv' with your actual CSV filename
csv_path = 'test_coordinates.csv'

gdf = sez.load_polygons(csv_path, crs="EPSG:4326")
all_grains = list(gdf.geometry)

plot the loaded grains on the image

In [ ]:
image = np.array(load_img(image_fname))
fig, ax = plt.subplots(figsize=(15,10))
ax.set_autoscale_on(False)
ax.set_xlim(0, image.shape[1])
ax.set_ylim(image.shape[0], 0)
plt.xticks([])
plt.yticks([])
seg.plot_image_w_colorful_grains(image, all_grains, ax, cmap='Paired')
plt.axis('equal')

now that all_grains is re-initialized and your grains are re-plotted, you are good to go back to adding and deleting grains!